In [1]:
#1-Defining the class which logs me in to the viewer website with more flexibility than fct used in other code
#CandidateViewerQuery and CandidateViewerRegistrar
import requests

class AutomationAPI:
    def __init__(self, base_url, survey_id, username, password):
        self.url = base_url
        self.survey_id = survey_id
        self.username = username
        self.password = password
        self.survey_aval = []
        self.session = requests.Session()

        # Login
        self._call("login", username=username, password=password)
        
        # Register endpoints
        self._register_endpoints()
        
        # Get list of surveys
        self.survey_aval = self._call("get_all_surveys")["surveys"]

        # Check if survey_id is valid
        if survey_id not in self.survey_aval:
            raise ValueError(f"Survey ID {survey_id} is not available. Available surveys: ", self.survey_aval)
        
        # Set survey
        self._call("set_survey", survey=survey_id)

    def _call(self, endpoint, **params):
        response = self.session.post(
            self.url,
            params={"endpoint": endpoint},  # GET
            data=params                      # POST
        )
        response.raise_for_status()
        
        try:
            data = response.json()
            if data["status"] != "success":
                raise RuntimeError(data["message"])
        except ValueError:
            raise RuntimeError("Invalid JSON response", response.text)
        
        return data["data"]

    def _register_endpoints(self):
        for endpoint in self._call("get_endpoints_aval")["endpoints"]:
            name = endpoint["name"]
            param_names = endpoint["params"]

            if hasattr(self, name):
                continue

            def make_method(endpoint_name, endpoint_params):
                def method(self, **kwargs):
                    missing = [p for p in endpoint_params if p not in kwargs]
                    if missing:
                        raise ValueError(f"Missing parameter(s): {', '.join(missing)}")
                    return self._call(endpoint_name, **kwargs)
                method.__name__ = endpoint_name
                method.__doc__  = f"Params: {', '.join(endpoint_params)}"
                return method

            setattr(self.__class__, name, make_method(name, param_names))

In [4]:
#2-Calling the function for the test folder
api = AutomationAPI(
    "https://sps.chimenet.ca/candidates/index.php?automation", "test", "Viewer Bot", "v4A13BNYwqU5okUZE^h9c&x*blzHrYMi"
)
#print(api.get_all_folders())
print(api.get_files(folder="test_21"))
#print(api.get_checked_files(folder="test_21"))

{'files': [{'file': 'Multi_Pointing_Groups_f_7.746_DM_107.877_6907f554999bcd1bafd30368', 'folder': 'test_21', 'remoteUrl': '/candidates/index.php?assets&file=Multi_Pointing_Groups_f_7.746_DM_107.877_6907f554999bcd1bafd30368&folder=test_21&type=candidate_image', 'remoteUrl_alt': '/candidates/index.php?assets&file=Multi_Pointing_Groups_f_7.746_DM_107.877_6907f554999bcd1bafd30368&folder=test_21&type=candidate_image_alt', 'checked': {'status': True, 'result': '<none>', 'date': '1769109445', 'by': 'Wenke Xia', 'rater_results': [], 'rating_consistency': {'consistent': None, 'status': 'no_enough_ratings', 'result': 'Not enough ratings yet.'}, 'info': [], 'tags': [{'id': '94', 'survey': 'test', 'folder': 'test_21', 'file': 'Multi_Pointing_Groups_f_7.746_DM_107.877_6907f554999bcd1bafd30368', 'tag': 'Test Tag', 'added_by': 'Viewer Bot', 'time': '1776103838', 'info': '[]', 'color': '#00BCD4', 'style': 'label-default', 'text': 'Test Tag'}]}}, {'file': 'Multi_Pointing_Groups_f_5.574_DM_50.903_69093

In [3]:
#3-Get the file details(including tag)
api.get_file_details(folder="test_21", file="Multi_Pointing_Groups_f_7.746_DM_107.877_6907f554999bcd1bafd30368")

{'status': {'file': 'Multi_Pointing_Groups_f_7.746_DM_107.877_6907f554999bcd1bafd30368',
  'folder': 'test_21',
  'remoteUrl': '/candidates/index.php?assets&file=Multi_Pointing_Groups_f_7.746_DM_107.877_6907f554999bcd1bafd30368&folder=test_21&type=candidate_image',
  'remoteUrl_alt': '/candidates/index.php?assets&file=Multi_Pointing_Groups_f_7.746_DM_107.877_6907f554999bcd1bafd30368&folder=test_21&type=candidate_image_alt',
  'checked': {'status': True,
   'result': '<none>',
   'date': '1769109445',
   'by': 'Wenke Xia',
   'rater_results': [],
   'rating_consistency': {'consistent': None,
    'status': 'no_enough_ratings',
    'result': 'Not enough ratings yet.'},
   'info': [],
   'tags': [{'id': '94',
     'survey': 'test',
     'folder': 'test_21',
     'file': 'Multi_Pointing_Groups_f_7.746_DM_107.877_6907f554999bcd1bafd30368',
     'tag': 'Test Tag',
     'added_by': 'Viewer Bot',
     'time': '1776103838',
     'info': '[]',
     'color': '#00BCD4',
     'style': 'label-default

In [14]:
#4-Function to get file info from specific rating
api.get_files_by_rating_type(folder="test_21", rating_type="new_candidates")
api.get_files_by_rating_type(folder="test_21", rating_type="faint")

{'files': [{'file': 'Multi_Pointing_Groups_f_28.890_DM_20.240_690b7d1b999bcd1baf29e75b',
   'folder': 'test_21',
   'remoteUrl': '/candidates/index.php?assets&file=Multi_Pointing_Groups_f_28.890_DM_20.240_690b7d1b999bcd1baf29e75b&folder=test_21&type=candidate_image',
   'remoteUrl_alt': '/candidates/index.php?assets&file=Multi_Pointing_Groups_f_28.890_DM_20.240_690b7d1b999bcd1baf29e75b&folder=test_21&type=candidate_image_alt',
   'checked': {'status': True,
    'result': '<faint>',
    'date': '1772653126',
    'by': 'Viewer Bot',
    'rater_results': {'Wenke Xia': {'result': '<none>', 'additional': False},
     'Viewer Admin': {'result': '<faint>', 'additional': False}},
    'rating_consistency': {'consistent': True,
     'status': 'faint_consistent',
     'result': '<faint>',
     'date': '1772653126'},
    'info': {'history': []},
    'tags': [{'id': '29',
      'survey': 'test',
      'folder': 'test_21',
      'file': 'Multi_Pointing_Groups_f_28.890_DM_20.240_690b7d1b999bcd1baf29e

In [9]:
#5-api.get_created_tags() function to create a tag
api.add_tag(folder="test_22", file="Multi_Pointing_Groups_f_7.746_DM_107.877_6907f554999bcd1bafd30368", tag="Test Tag")
api.get_file_details(folder="test_22", file="Multi_Pointing_Groups_f_7.746_DM_107.877_6907f554999bcd1bafd30368")

{'status': {'file': 'Multi_Pointing_Groups_f_7.746_DM_107.877_6907f554999bcd1bafd30368',
  'folder': 'test_22',
  'remoteUrl': '/candidates/index.php?assets&file=Multi_Pointing_Groups_f_7.746_DM_107.877_6907f554999bcd1bafd30368&folder=test_22&type=candidate_image',
  'remoteUrl_alt': '/candidates/index.php?assets&file=Multi_Pointing_Groups_f_7.746_DM_107.877_6907f554999bcd1bafd30368&folder=test_22&type=candidate_image_alt',
  'checked': {'status': True,
   'result': '',
   'date': '1775624069',
   'by': '',
   'rater_results': {'Wenke Xia': {'result': '<none>', 'additional': False}},
   'rating_consistency': {'consistent': None,
    'status': 'no_enough_ratings',
    'result': 'Not enough ratings yet.'},
   'info': [],
   'tags': [{'id': '131',
     'survey': 'test',
     'folder': 'test_22',
     'file': 'Multi_Pointing_Groups_f_7.746_DM_107.877_6907f554999bcd1bafd30368',
     'tag': 'Test Tag',
     'added_by': 'Viewer Bot',
     'time': '1778076755',
     'info': '[]',
     'color':

In [11]:
#6-api.deltete_tags() function to delete a tag
api.delete_tag(folder="test_22", file="Multi_Pointing_Groups_f_7.746_DM_107.877_6907f554999bcd1bafd30368", tag="Test Tag")
api.get_file_details(folder="test_22", file="Multi_Pointing_Groups_f_7.746_DM_107.877_6907f554999bcd1bafd30368")

{'status': {'file': 'Multi_Pointing_Groups_f_7.746_DM_107.877_6907f554999bcd1bafd30368',
  'folder': 'test_22',
  'remoteUrl': '/candidates/index.php?assets&file=Multi_Pointing_Groups_f_7.746_DM_107.877_6907f554999bcd1bafd30368&folder=test_22&type=candidate_image',
  'remoteUrl_alt': '/candidates/index.php?assets&file=Multi_Pointing_Groups_f_7.746_DM_107.877_6907f554999bcd1bafd30368&folder=test_22&type=candidate_image_alt',
  'checked': {'status': True,
   'result': '',
   'date': '1775624069',
   'by': '',
   'rater_results': {'Wenke Xia': {'result': '<none>', 'additional': False}},
   'rating_consistency': {'consistent': None,
    'status': 'no_enough_ratings',
    'result': 'Not enough ratings yet.'},
   'info': [],
   'tags': []}}}

In [15]:
#7- Example for single day fold(daily cand folder):
api = AutomationAPI(
    "https://sps.chimenet.ca/candidates/index.php?automation", "dailycands", "Viewer Bot", "v4A13BNYwqU5okUZE^h9c&x*blzHrYMi"
)

api.get_files(folder="2026-03-26")["files"][0] # the [0] select the first candidate in the list of n cand

{'file': 'Multi_Pointing_Groups_f_5.112_DM_8.804_69ca90a526d8637446f5b0dd',
 'folder': '2026-03-26',
 'remoteUrl': '/candidates/index.php?assets&file=Multi_Pointing_Groups_f_5.112_DM_8.804_69ca90a526d8637446f5b0dd&folder=2026-03-26&type=candidate_image',
 'remoteUrl_alt': '/candidates/index.php?assets&file=Multi_Pointing_Groups_f_5.112_DM_8.804_69ca90a526d8637446f5b0dd&folder=2026-03-26&type=candidate_image_alt',
 'checked': {'status': True,
  'result': 'B0655+64',
  'date': '1774990714',
  'by': 'Viewer Bot',
  'rater_results': {'Habtamu Menberu Tedila': {'result': 'B0655+64',
    'additional': False},
   'Reynier Squillace': {'result': 'B0655+64', 'additional': False}},
  'rating_consistency': {'consistent': True,
   'status': 'consistent',
   'result': 'B0655+64',
   'date': '1774990714'},
  'info': {'history': []},
  'tags': []}}